# Proyecto de aula · Principios de Inteligencia Artificial 2026-B

**Plan de repaso semanal** — cuaderno individual del corte 3.

> **Estado del borrador:** el programa y las 12 pruebas funcionan y ya están ejecutados. Los textos marcados con **[COMPLETAR]** son los que solo usted puede escribir (su nombre, lo que hizo usted, los registros reales de la bitácora). No los deje en blanco ni los invente: la guía considera falta inventar registros.
>
> Antes de entregar use «Entorno de ejecución · Reiniciar y ejecutar todo» y confirme que no hay errores.

## 1. Ficha del proyecto

| Campo | Respuesta |
|---|---|
| Nombre completo | Andrés Felipe Bolívar Arias |
| Título del proyecto | Plan de repaso semanal: reparte las horas libres de un estudiante entre sus materias según la dificultad y la fecha de cada examen |
| Problema | Un estudiante con varios exámenes cercanos no sabe a qué materia dedicarle cada hora libre y tiende a estudiar lo que le gusta o lo que tiene más a mano, dejando para el final lo urgente o lo difícil |
| Usuario | Un estudiante universitario con 2 a 6 materias con examen en las próximas semanas |
| Qué entra | Lista de materias (nombre, fecha del examen, dificultad 1 a 3) y las horas libres que tiene cada día de la semana. Datos ficticios o propios, sin información de otras personas |
| Qué sale | Un horario de 7 días que dice qué materia estudiar en cada hora libre, más el total de horas por materia |

## 2. Cómo se usa

1. Ejecute las celdas de la parte 3 con el botón ▶ (o «Entorno de ejecución · Ejecutar todo»). Solo definen las funciones; no imprimen nada.
2. En la celda de uso (la que empieza con `hoy = ...`) cambie la lista `mis_materias`: cada materia es `("nombre", "AAAA-MM-DD", dificultad)` con dificultad 1 (fácil), 2 (media) o 3 (difícil). Cambie también `mis_horas` con las horas libres de cada día.
3. Ejecute esa celda. Aparece el plan de la semana, un día por línea, y el total de horas por materia. Si algún dato está mal (fecha mal escrita, dificultad 5, etc.) el programa dice qué corregir.

Para usarlo con la fecha real escriba `hoy = date.today()` en lugar de la fecha fija del ejemplo.

## 3. El programa

Reglas del programa, en pocas palabras:

- **Peso** de cada materia = dificultad × urgencia (urgencia 3 si el examen es en 3 días o menos, 2 si es en 7 días o menos, 1 si es más lejos).
- Cada materia tiene una **meta** de horas proporcional a su peso.
- Cada hora libre se le da a la materia a la que más horas le faltan para su meta (si empatan, a la del examen más cercano).
- Máximo 2 horas seguidas de la misma materia por día, y no se estudia una materia después de su examen.

**[COMPLETAR]** Los comentarios del código deben estar en sus propias palabras: léalos uno por uno y reescriba los que no pueda explicar.

In [1]:
# PLAN DE REPASO SEMANAL
# Recibe las materias (con fecha de examen y dificultad) y las horas libres
# de cada día, y devuelve qué materia estudiar en cada hora libre de la semana.
from datetime import date, timedelta

DIAS = ["lunes", "martes", "miercoles", "jueves", "viernes", "sabado", "domingo"]
NOMBRES_DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]  # para mostrar
MAX_HORAS_POR_MATERIA_AL_DIA = 2  # para no pasar todo el día con una sola materia


def sin_tildes(texto):
    """Pasa a minúsculas y quita tildes para aceptar 'Miércoles' o 'miercoles'."""
    cambios = str.maketrans("áéíóú", "aeiou")
    return texto.strip().lower().translate(cambios)


def validar_datos(materias, horas_libres, hoy):
    """Revisa los datos. Si algo está mal lanza ValueError con un mensaje claro.
    Si todo está bien devuelve las materias listas y las horas libres por día (0 a 6)."""
    if len(materias) == 0:
        raise ValueError("No hay materias: agregue al menos una.")

    listas = []
    nombres_vistos = []
    for nombre, fecha_texto, dificultad in materias:
        if not str(nombre).strip():
            raise ValueError("Hay una materia sin nombre.")
        if nombre in nombres_vistos:
            raise ValueError(f"La materia {nombre} está repetida: use nombres distintos.")
        nombres_vistos.append(nombre)
        try:
            fecha = date.fromisoformat(fecha_texto)  # espera AAAA-MM-DD
        except (ValueError, TypeError):
            raise ValueError(f"La fecha de {nombre} no es válida: use AAAA-MM-DD.")
        if fecha < hoy:
            raise ValueError(f"El examen de {nombre} ya pasó ({fecha_texto}).")
        if dificultad not in (1, 2, 3):
            raise ValueError(f"La dificultad de {nombre} debe ser 1, 2 o 3.")
        listas.append((nombre, fecha, dificultad))

    horas = {i: 0 for i in range(7)}
    for dia, cantidad in horas_libres.items():
        dia_limpio = sin_tildes(dia)
        if dia_limpio not in DIAS:
            raise ValueError(f"'{dia}' no es un día de la semana.")
        if not isinstance(cantidad, (int, float)) or cantidad < 0 or cantidad > 16:
            raise ValueError(f"Las horas del {dia} deben estar entre 0 y 16.")
        horas[DIAS.index(dia_limpio)] = int(cantidad)  # 1,5 h cuenta como 1 h

    if sum(horas.values()) == 0:
        raise ValueError("No hay horas libres: escriba al menos una hora en algún día.")
    return listas, horas


In [2]:
def calcular_pesos(materias, hoy):
    """Peso = dificultad x urgencia. Entre más difícil y más cerca el examen, más peso."""
    pesos = {}
    for nombre, fecha, dificultad in materias:
        dias_que_faltan = (fecha - hoy).days
        if dias_que_faltan <= 3:
            urgencia = 3
        elif dias_que_faltan <= 7:
            urgencia = 2
        else:
            urgencia = 1
        pesos[nombre] = dificultad * urgencia
    return pesos


def armar_plan(materias, horas_libres, hoy):
    """Reparte las horas libres de los 7 días que empiezan en 'hoy'.
    Devuelve una lista con un diccionario por día: fecha, nombre del día y las
    materias asignadas hora por hora. Las horas sin materia posible quedan vacías."""
    materias, horas = validar_datos(materias, horas_libres, hoy)
    pesos = calcular_pesos(materias, hoy)

    # Meta: cuántas horas debería recibir cada materia según su peso.
    total_horas = sum(horas.values())
    suma_pesos = sum(pesos.values())
    metas = {n: total_horas * p / suma_pesos for n, p in pesos.items()}
    asignadas = {n: 0 for n in pesos}

    plan = []
    for i in range(7):
        dia = hoy + timedelta(days=i)
        # Solo se estudia una materia hasta el día de su examen.
        disponibles = [m for m in materias if m[1] >= dia]
        bloques = []
        por_dia = {}
        for _ in range(horas[dia.weekday()]):
            if not disponibles:
                break  # ya pasaron todos los exámenes: esa hora queda libre
            # Primero se descartan las que ya tienen su máximo de horas hoy.
            candidatas = [m for m in disponibles
                          if por_dia.get(m[0], 0) < MAX_HORAS_POR_MATERIA_AL_DIA]
            if not candidatas:
                candidatas = disponibles  # si no queda otra, se permite repetir
            # Gana la materia a la que más horas le faltan; si empatan, la del examen más cercano.
            elegida = max(candidatas,
                          key=lambda m: (round(metas[m[0]] - asignadas[m[0]], 6), -m[1].toordinal()))
            nombre = elegida[0]
            bloques.append(nombre)
            asignadas[nombre] += 1
            por_dia[nombre] = por_dia.get(nombre, 0) + 1
        plan.append({"fecha": dia, "dia": NOMBRES_DIAS[dia.weekday()], "bloques": bloques})
    return plan


In [3]:
def horas_por_materia(plan):
    """Cuenta cuántas horas quedaron en total para cada materia."""
    totales = {}
    for dia in plan:
        for nombre in dia["bloques"]:
            totales[nombre] = totales.get(nombre, 0) + 1
    return totales


def texto_plan(plan):
    """Convierte el plan en un texto fácil de leer, un día por línea."""
    lineas = []
    for dia in plan:
        if not dia["bloques"]:
            detalle = "sin horas de estudio"
        else:
            # Une las horas seguidas de la misma materia: 'Cálculo (2 h)'.
            partes = []
            for nombre in dia["bloques"]:
                if partes and partes[-1][0] == nombre:
                    partes[-1][1] += 1
                else:
                    partes.append([nombre, 1])
            detalle = " + ".join(f"{n} ({h} h)" for n, h in partes)
        lineas.append(f"{dia['dia']:<10} {dia['fecha']}: {detalle}")
    return "\n".join(lineas)


### Uso del programa
Cambie los datos para probarlo con sus materias.

In [4]:
# Aquí se usa el programa. Cambie los datos por los suyos (pueden ser inventados).
# Cada materia es: (nombre, fecha del examen AAAA-MM-DD, dificultad 1=fácil, 2=media, 3=difícil)
hoy = date(2026, 11, 23)  # para usar la fecha real escriba: hoy = date.today()

mis_materias = [
    ("Cálculo II", "2026-11-27", 3),
    ("Programación", "2026-11-26", 2),
    ("Física", "2026-12-02", 2),
    ("Inglés", "2026-12-04", 1),
]

# Horas libres por día para estudiar (si un día no aparece, cuenta como 0).
mis_horas = {"lunes": 2, "martes": 2, "miércoles": 1, "jueves": 2,
             "viernes": 2, "sábado": 4, "domingo": 3}

try:
    plan = armar_plan(mis_materias, mis_horas, hoy)
    print("PLAN DE REPASO DE LA SEMANA\n")
    print(texto_plan(plan))
    print("\nTotal de horas por materia:", horas_por_materia(plan))
except ValueError as error:
    print("Revise los datos:", error)


PLAN DE REPASO DE LA SEMANA

Lunes      2026-11-23: Programación (1 h) + Cálculo II (1 h)
Martes     2026-11-24: Programación (1 h) + Cálculo II (1 h)
Miércoles  2026-11-25: Programación (1 h)
Jueves     2026-11-26: Cálculo II (1 h) + Programación (1 h)
Viernes    2026-11-27: Cálculo II (2 h)
Sábado     2026-11-28: Física (2 h) + Inglés (2 h)
Domingo    2026-11-29: Física (2 h) + Inglés (1 h)

Total de horas por materia: {'Programación': 4, 'Cálculo II': 5, 'Física': 4, 'Inglés': 3}


## 4. Pruebas

12 casos: 4 normales y 8 difíciles (datos raros, vacíos o imposibles). Antes de correrlas se calculó a mano el resultado esperado de cada una; la celda compara lo esperado con lo obtenido.

In [5]:
# Cada prueba tiene: descripción, datos de entrada (materias, horas libres) y resultado esperado.
# Para las pruebas se fija el lunes 23 de noviembre de 2026 como "hoy".
HOY_PRUEBA = date(2026, 11, 23)
LEJOS = "2026-12-30"  # un examen lejano, para que no influya la urgencia


def probar(materias, horas):
    """Corre el programa y devuelve las horas por materia, o el mensaje de error."""
    try:
        return horas_por_materia(armar_plan(materias, horas, HOY_PRUEBA))
    except ValueError as error:
        return str(error)


pruebas = [
    ("Normal: dos materias iguales se reparten por igual",
     ([("A", LEJOS, 2), ("B", LEJOS, 2)], {"lunes": 4, "martes": 2}),
     {"A": 3, "B": 3}),
    ("Normal: la materia difícil y urgente recibe todas las horas hasta su examen",
     ([("Cálculo", "2026-11-25", 3), ("Historia", LEJOS, 1)], {d: 2 for d in DIAS}),
     {"Cálculo": 6, "Historia": 8}),
    ("Normal: máximo 2 horas de una misma materia en un día",
     ([("A", LEJOS, 3), ("B", LEJOS, 1)], {"lunes": 4}),
     {"A": 2, "B": 2}),
    ("Normal: 1,5 horas libres cuentan como 1 hora",
     ([("A", LEJOS, 2)], {"lunes": 1.5}),
     {"A": 1}),
    ("Caso difícil: examen hoy, solo se estudia hoy",
     ([("A", "2026-11-23", 2), ("B", LEJOS, 2)], {"lunes": 2, "martes": 2}),
     {"A": 2, "B": 2}),
    ("Caso difícil: sin materias",
     ([], {"lunes": 2}),
     "No hay materias: agregue al menos una."),
    ("Caso difícil: dificultad imposible",
     ([("Cálculo", LEJOS, 5)], {"lunes": 2}),
     "La dificultad de Cálculo debe ser 1, 2 o 3."),
    ("Caso difícil: examen que ya pasó",
     ([("Cálculo", "2026-11-20", 2)], {"lunes": 2}),
     "El examen de Cálculo ya pasó (2026-11-20)."),
    ("Caso difícil: fecha mal escrita",
     ([("Cálculo", "27/11/2026", 2)], {"lunes": 2}),
     "La fecha de Cálculo no es válida: use AAAA-MM-DD."),
    ("Caso difícil: materia repetida",
     ([("Cálculo", LEJOS, 2), ("Cálculo", LEJOS, 3)], {"lunes": 2}),
     "La materia Cálculo está repetida: use nombres distintos."),
    ("Caso difícil: día que no existe",
     ([("Cálculo", LEJOS, 2)], {"funday": 2}),
     "'funday' no es un día de la semana."),
    ("Caso difícil: ninguna hora libre",
     ([("Cálculo", LEJOS, 2)], {"lunes": 0}),
     "No hay horas libres: escriba al menos una hora en algún día."),
]

correctas = 0
for descripcion, (materias, horas), esperado in pruebas:
    obtenido = probar(materias, horas)
    estado = "CORRECTO" if obtenido == esperado else "REVISAR"
    correctas += estado == "CORRECTO"
    print(f"{estado}: {descripcion}\n    esperado: {esperado}\n    obtenido: {obtenido}")
print(f"\n{correctas} de {len(pruebas)} pruebas correctas")


CORRECTO: Normal: dos materias iguales se reparten por igual
    esperado: {'A': 3, 'B': 3}
    obtenido: {'A': 3, 'B': 3}
CORRECTO: Normal: la materia difícil y urgente recibe todas las horas hasta su examen
    esperado: {'Cálculo': 6, 'Historia': 8}
    obtenido: {'Cálculo': 6, 'Historia': 8}
CORRECTO: Normal: máximo 2 horas de una misma materia en un día
    esperado: {'A': 2, 'B': 2}
    obtenido: {'A': 2, 'B': 2}
CORRECTO: Normal: 1,5 horas libres cuentan como 1 hora
    esperado: {'A': 1}
    obtenido: {'A': 1}
CORRECTO: Caso difícil: examen hoy, solo se estudia hoy
    esperado: {'A': 2, 'B': 2}
    obtenido: {'A': 2, 'B': 2}
CORRECTO: Caso difícil: sin materias
    esperado: No hay materias: agregue al menos una.
    obtenido: No hay materias: agregue al menos una.
CORRECTO: Caso difícil: dificultad imposible
    esperado: La dificultad de Cálculo debe ser 1, 2 o 3.
    obtenido: La dificultad de Cálculo debe ser 1, 2 o 3.
CORRECTO: Caso difícil: examen que ya pasó
    esperad

**Qué aprendí de las pruebas:** las 12 pruebas dan CORRECTO. La prueba de materia repetida existe porque, al revisar casos límite, se vio que dos materias con el mismo nombre se mezclaban en silencio (ver parte 7). Las pruebas con examen «hoy» y con examen a mitad de semana comprueban que no se asignan horas a una materia después de su examen.

**[COMPLETAR]** Escriba con sus palabras qué le sorprendió o qué cambiaría después de ejecutar usted mismo las pruebas en Colab.

## 5. Bitácora de prompts

Mínimo 8 registros. **Hay 2 registros reales (n.º 1 y 2).** Los registros 3 a 8 deben escribirse a medida que usted use la IA de verdad (por ejemplo: pedirle que le explique una celda, pedirle una función nueva, pegarle un error). Copie cada prompt tal como lo escribió. Inventar registros es una falta según la guía.

| N.º | Fecha | Herramienta | Prompt | Resumen de la respuesta | Qué hice (usé, cambié o descarté) | Cómo lo comprobé |
|---|---|---|---|---|---|---|
| 1 | 2026-11-11 | Claude Code (Claude) | Ver «Prompt 1» abajo | Leyó la guía, la plantilla y el FO-IV-159; propuso el proyecto «Plan de repaso semanal»; escribió el programa, 12 pruebas y un borrador de las diez partes del cuaderno y del formato | **[COMPLETAR]** | Claude ejecutó el programa y las 12 pruebas (todas CORRECTO) y encontró y corrigió 2 fallos (parte 7). **[COMPLETAR]** Falta que usted lo ejecute en Colab y explique cada celda |
| 2 | 2026-11-11 | Claude Code (Claude) | yo debo que explicarlo | Explicó el programa celda por celda, con un ejemplo numérico (prueba 2), posibles preguntas del docente y ejercicios para practicar | **[COMPLETAR]** | **[COMPLETAR]** Por ejemplo: seguir el ejemplo en Colab cambiando una fecha o `MAX_HORAS_POR_MATERIA_AL_DIA` y ver si el resultado coincide con lo explicado |
| 3 | | | | | | |
| 4 | | | | | | |
| 5 | | | | | | |
| 6 | | | | | | |
| 7 | | | | | | |
| 8 | | | | | | |

**Prompt 1** (copiado tal como se escribió; se adjuntaron la guía en PDF, la plantilla del cuaderno y el formato FO-IV-159):

> Proyecto de aula del corte 3, individual. Cada estudiante construye en un cuaderno de Google Colab algo pequeño que funcione y le sirva a una persona real, con ayuda de un asistente de IA. Además, documenta cómo usó esa ayuda: qué le pidió, cómo mejoró sus prompts, qué errores encontró, cómo los comprobó y cómo la usó de manera responsable.
>
> Los avances se revisan en clase el miércoles 11, el viernes 13 y el miércoles 18 de noviembre, y no tienen nota por separado. Para cada avance, deje en el texto en línea el enlace de su cuaderno en Colab. La entrega final y la sustentación son el viernes 20 de noviembre; se recibe hasta las 23:59.
>
> En la entrega final escriba en el texto en línea su nombre y el enlace del cuaderno, y suba un ZIP con el cuaderno .ipynb ejecutado y el FO-IV-159 diligenciado. Adjuntos encontrará la guía en PDF, con las ideas de proyecto, cómo gestionar los prompts y la rúbrica; la plantilla del cuaderno y el formato FO-IV-159.
>
> Se recomienda empezar por un problema sencillo, llenar la bitácora mientras trabaja y no entregar nada que no pueda explicar.

## 6. Mejoras de prompt

Mínimo 2 casos. **Pendiente: no hay casos reales todavía.** Un prompt como el n.º 1 fue de una sola vez, sin mejora posterior, así que no se registra aquí como mejora.

Cómo conseguir dos casos reales: pídale a la IA una función nueva para el programa (por ejemplo, un día de descanso obligatorio) con un prompt corto como «agrégame un día de descanso». Guarde la respuesta, anote qué salió mal y vuelva a pedirlo con rol, contexto, tarea, formato, ejemplo y límites. Repita con otra función.

### Mejora 1

- **Prompt anterior:** [COMPLETAR]
- **Qué salió mal:** [COMPLETAR]
- **Prompt mejorado:** [COMPLETAR]
- **Qué agregué** (rol, contexto, tarea, formato, ejemplo o límites): [COMPLETAR]
- **Qué mejoró en la respuesta:** [COMPLETAR]

### Mejora 2

- **Prompt anterior:** [COMPLETAR]
- **Qué salió mal:** [COMPLETAR]
- **Prompt mejorado:** [COMPLETAR]
- **Qué agregué** (rol, contexto, tarea, formato, ejemplo o límites): [COMPLETAR]
- **Qué mejoró en la respuesta:** [COMPLETAR]

## 7. Errores y comprobación

Dos errores reales del primer borrador que escribió la IA. Ambos los encontró la propia IA al ejecutar y revisar su código en la sesión del registro n.º 1. **[COMPLETAR]** Repita usted la comprobación (ver la última columna) y anote si obtuvo lo mismo.

| N.º | Qué dijo o hizo la IA | Por qué estaba mal o era dudoso | Cómo lo detecté | Cómo lo corregí |
|---|---|---|---|---|
| 1 | Escribió el horario mostrando los días como «Miercoles» y «Sabado», sin tilde | El código usaba la misma lista de días (sin tildes, para comparar datos) para escribir el resultado; un usuario hispanohablante lo vería mal escrito | Al ejecutar el programa y leer la salida impresa | Se separaron dos listas: `DIAS` (sin tildes, para comparar) y `NOMBRES_DIAS` (con tildes, para mostrar) |
| 2 | Permitía dos materias con el mismo nombre, por ejemplo `("Cálculo", ..., 2)` y `("Cálculo", ..., 3)` | Los pesos y las horas se guardan en diccionarios con el nombre como clave: la segunda materia pisaba a la primera y el plan salía mal sin ningún aviso | Al revisar casos límite pensando qué datos rompen el programa (nombres repetidos, fechas mal escritas, cero horas) | Se agregó una validación que lanza el error «La materia X está repetida» y la prueba «Caso difícil: materia repetida». Para verlo fallar: quite esa validación y corra la prueba |

## 8. Uso responsable

Respuestas sobre **este** proyecto. Los textos son un borrador basado en lo que realmente pasó; **[COMPLETAR]** revíselos, ajústelos y agregue lo que usted hizo de verdad.

1. **Comprobación:** el resultado esperado de cada prueba se calculó a mano antes de ejecutarla, no se copió de la salida del programa; las 12 pruebas coinciden. También se revisaron casos límite (datos vacíos, fechas inválidas, nombres repetidos), lo que destapó el error n.º 2. **[COMPLETAR]** Anote si usted ejecutó todo en Colab y qué comprobó por su cuenta.
2. **Datos:** al asistente solo se le dieron las instrucciones de la tarea y archivos de la clase. No se compartieron nombre, cédula, correo, notas ni horarios reales. Las materias, fechas y horas del ejemplo son ficticias. **[COMPLETAR]** Si usted prueba con sus propias materias, eso ya son datos suyos: decida qué comparte con la IA.
3. **Sesgo:** el programa no usa datos de personas ni decide sobre ellas, así que no clasifica grupos. Su riesgo es otro: la «dificultad» es la opinión del propio estudiante, y una materia que parece fácil pero se subestima recibe pocas horas. Además no considera descanso, sueño ni trabajo. Se revisó con el ejemplo del plan semanal: la materia de dificultad 1 y examen lejano recibe pocas horas, y así debe pasar solo si el estudiante la calificó bien. **[COMPLETAR]**
4. **Fuentes:** la IA propuso tres referencias para el FO-IV-159. Dos se comprobaron con una búsqueda en la web: Dunlosky et al. (2013), monografía en *Psychological Science in the Public Interest*, 14(1), 4–58 (DOI 10.1177/1529100612453266), y la Ley Estatutaria 1581 de 2012 (Diario Oficial 48.587). La tercera, Russell y Norvig (2021), *Artificial Intelligence: A Modern Approach* (4.ª ed., Pearson), no se comprobó en esta sesión. **[COMPLETAR]** Compruebe usted las tres antes de entregar.
5. **Autoría:** la IA escribió la primera versión del programa, de las pruebas y de los textos de este cuaderno. **[COMPLETAR]** Escriba qué parte hizo usted (por ejemplo: ejecutarlo en Colab, cambiar los datos, escribir los comentarios con sus palabras, hacer los registros 2 a 8) y confirme que puede explicar cada celda en la sustentación.
6. **Límites:** sirve para organizar horas de estudio con reglas simples; no sirve para decidir sobre salud, becas, notas o personas, ni para planear más horas de las que el estudiante puede sostener. Si el plan le parece irreal (por ejemplo, ninguna hora para una materia que ya va mal), debe decidir una persona: el propio estudiante, un profesor o un tutor. El plan es una sugerencia, no una orden.

## 9. Declaración de uso de IA

| Herramienta | Para qué la usé | Qué hice yo |
|---|---|---|
| Claude Code (Claude) | Escribir la primera versión del programa, de las 12 pruebas y de los textos del cuaderno y del FO-IV-159; revisar su propio código | **[COMPLETAR: por ejemplo, la ejecuté en Colab, revisé y expliqué cada celda, cambié los datos, escribí los registros 2 a 8]** |

Declaro que esta información es verdadera y que puedo explicar todo lo que entrego. **Nombre:** [COMPLETAR: escriba su nombre]

## 10. Summary

Many university students have several exams in the same weeks and do not know which subject to study in each free hour. This notebook contains a small Python tool, built with the help of an AI assistant, that turns a list of subjects (with exam dates and a difficulty from 1 to 3) and the free hours of each weekday into a seven-day study plan. The tool gives more hours to subjects that are harder and have closer exams, limits any subject to two hours per day, and stops assigning a subject after its exam. The main result is that the program passed all 12 tests, including difficult cases such as empty data, wrong dates and repeated subject names, and the AI-written first draft needed two fixes that were found by running and reviewing it. One important limitation is that the difficulty is the student's own opinion, and the plan ignores rest, sleep and work, so it is only a suggestion and a person must decide.